# Feature selection by category: shared core

Shared logic for the category notebooks (`01_claims_...` to `06_comorb_claims_...`). Do not run this notebook on its own: each category notebook installs LightGBM, sets its **category config** (`CATEGORY`, `CATEGORY_LABEL`, `ADD_CALENDAR`, `SOURCE_TABLES`, paths) and then runs this notebook with `%run`, so every category uses exactly the same method.

**Part 1 (NBA 3.1 method):** writer definition (`TOTAL_HCP_NBRX_CLAIMS_KERENDIA_IN_LAST_12_MONTH` > 0, empty = 0), two data sets (writers / non-writers), mean / median analysis ranking features by how well they separate writers from non-writers (`DISCRIMINATION` = \|2 x AUC - 1\|).

**Part 2 (per group):** NBA 3.1 clean-up (removal stages 1-4, flags 5-6), one over-fitted LightGBM Poisson model on all chosen cohorts, SHAP values, best features.

**Widgets:** `RUN_NAME`, `SAMPLE_LIST` (empty = full HCP universe, weight 1), `COHORTS` (e.g. `c12-c24`; default `c1-c28`), `N_THREADS` (0 = all driver cores, for the numba statistics in steps 6-7), `TEST_HCPS` / `MAX_FEATURES` (0 = all), `N_TREES`, `LEARNING_RATE`, `NUM_LEAVES`, `MIN_DATA_IN_LEAF`, `TOP_N`. Outputs: `hcp_level_model/dev_pipeline/notebook_output/feature_selection_by_category/<RUN_NAME>/<CATEGORY>/`.

In [0]:
_missing_cfg = [v for v in ("CATEGORY", "CATEGORY_LABEL", "ADD_CALENDAR", "SOURCE_TABLES", "VOL31", "NO", "DEV") if v not in globals()]
assert not _missing_cfg, f"run this notebook from a category notebook (%run) after its config cell; missing: {_missing_cfg}"

import numpy as np, pandas as pd, re, os, time, json
from scipy.stats import rankdata
from pyspark.sql import functions as F
import lightgbm as lgb
import matplotlib.pyplot as plt

dbutils.widgets.text("RUN_NAME", "run_2026_11_26")
dbutils.widgets.text("SAMPLE_LIST", VOL31 + "hcp_level_model/dev_pipeline/sample_lists/option_a_10k_hcps_10_months")
dbutils.widgets.text("COHORTS", "c1-c28")    # cohort range(s), e.g. "c12-c24" or "c1-c5,c27"
dbutils.widgets.text("N_THREADS", "0")       # threads for the per-feature statistics in steps 6-7 (0 = all driver cores)
dbutils.widgets.text("TEST_HCPS", "0")        # N = quick test on N HCPs (same group mix as the sample)
dbutils.widgets.text("MAX_FEATURES", "0")     # N = quick test on N features spread over the tables
dbutils.widgets.text("N_TREES", "500")
dbutils.widgets.text("LEARNING_RATE", "0.05")
dbutils.widgets.text("NUM_LEAVES", "63")
dbutils.widgets.text("MIN_DATA_IN_LEAF", "20")
dbutils.widgets.text("TOP_N", "50")

RUN_NAME         = dbutils.widgets.get("RUN_NAME").strip()
SAMPLE_LIST      = dbutils.widgets.get("SAMPLE_LIST").strip().rstrip("/")
def parse_cohorts(spec):
    out = []
    for part in spec.replace(" ", "").split(","):
        a, _, b = part.partition("-")
        a = int(a.lower().lstrip("c")); b = int(b.lower().lstrip("c")) if b else a
        out += [f"c{i}" for i in range(a, b + 1)]
    assert out and len(set(out)) == len(out) and all(1 <= int(c[1:]) <= 28 for c in out), f"bad COHORTS: {spec}"
    return out
COHORT_LIST      = parse_cohorts(dbutils.widgets.get("COHORTS").strip() or "c1-c28")
N_THREADS        = int(dbutils.widgets.get("N_THREADS") or 0) or os.cpu_count()
TEST_HCPS        = int(dbutils.widgets.get("TEST_HCPS") or 0)
MAX_FEATURES     = int(dbutils.widgets.get("MAX_FEATURES") or 0)
N_TREES          = int(dbutils.widgets.get("N_TREES"))
LEARNING_RATE    = float(dbutils.widgets.get("LEARNING_RATE"))
NUM_LEAVES       = int(dbutils.widgets.get("NUM_LEAVES"))
MIN_DATA_IN_LEAF = int(dbutils.widgets.get("MIN_DATA_IN_LEAF"))
TOP_N            = int(dbutils.widgets.get("TOP_N"))
assert RUN_NAME, "RUN_NAME is required"

# ---- Inputs ----
UNIVERSE_PATH = VOL31 + "HCP_Universe_without_call_hcps"
COHORTS_PATH  = VOL31 + "cohorts/claims/cohorts.parquet"
TARGET_PATH   = NO + "target_variable/target_variable"
WRITER_PATH   = DEV + "hcp_features_v7_cohort_aligned/nbrx_features"
WRITER_COL    = "TOTAL_HCP_NBRX_CLAIMS_KERENDIA_IN_LAST_12_MONTH"
TRAINING_COHORTS = [f"c{i}" for i in range(1, 25)]   # only for the median fill rule, as in Combine

# ---- Outputs ----
TEST    = bool(TEST_HCPS or MAX_FEATURES)
OUT_DIR = (f"{DEV}feature_selection_by_category/{RUN_NAME}/{CATEGORY}/"
           + (f"_test_{TEST_HCPS}_hcps_{MAX_FEATURES}_features/" if TEST else ""))
KEYS_PATH = OUT_DIR + "keys"            # HCP x cohort rows, target, writer flag, weight
DATA_PATH = OUT_DIR + "category_data"   # same rows, this category's features (filled)
assert OUT_DIR.startswith(VOL31) and "kerendia_nba_3_volume" not in OUT_DIR, f"refusing to write outside the 3.1 volume: {OUT_DIR}"

# ---- NBA 3.1 clean-up thresholds (02_nba_3_1_statistical_cleanup) ----
NEAR_CONSTANT_SHARE = 0.995   # stage 2
MIN_FILL_SHARE      = 0.01    # stage 3
NEAR_COPY_RHO       = 0.95    # stage 4
NOISE_QUANTILE      = 0.99    # flag 5 noise bar
N_BINS              = 10      # correlation ratio bins
COPY_SAMPLE_ROWS    = 20000   # stage 4 rows
CHUNK               = 500     # max features per pass (statistics); fewer when there are many rows (about 1e9 values per pass)
DISC_CHUNK          = 500     # max features per pass (mean / median analysis)
SHAP_CHUNK          = 10000   # rows per SHAP pass
SHAP_DIRECTION_ROWS = 20000   # rows for the sign of each feature's effect
SEED                = 42

KEY_COLS = {"HCP_COHORT_ID", "PROVIDER_ID", "COHORT_ID", "COHORT"}
METADATA_COLS = {"COHORT_MONTH", "COHORT_SET", "CUTOFF_DATE", "DATASET", "OFFSET_DAYS", "PULL_DT",
                 "SVC_DT", "SVC_DT_M", "PREDICTION_MONTH", "LOOKBACK_START_DATE", "LOOKBACK_END_DATE",
                 "SAMPLE_CUTOFF_DATE", "ZIP5", "DMA_CODE", "STATE_NAME"}
GROUPS = {"writers": True, "non_writers": False}
STAGES = {"1_constant": "same value in every row",
          "2_near_constant": f"one value in at least {NEAR_CONSTANT_SHARE:.1%} of rows",
          "3_low_fill": f"non-default value in under {MIN_FILL_SHARE:.0%} of rows",
          "4_near_copy": f"rank correlation at least {NEAR_COPY_RHO} with a stronger kept feature"}
FLAGS  = {"FLAG_5_NO_SIGNAL": "linear and non-linear link to NBRx both below the noise bar",
          "FLAG_6_UNSTABLE": "link to NBRx not repeated in both halves of the doctors (features with a signal only)"}
print(f"CATEGORY={CATEGORY} ({CATEGORY_LABEL})  RUN_NAME={RUN_NAME}  TEST_HCPS={TEST_HCPS}  MAX_FEATURES={MAX_FEATURES}")
print(f"Model: {N_TREES} trees, learning rate {LEARNING_RATE}, {NUM_LEAVES} leaves, min {MIN_DATA_IN_LEAF} rows per leaf; {os.cpu_count()} driver cores")
print("Sample list:", SAMPLE_LIST or "- (full HCP universe)")
print(f"Cohorts: {COHORT_LIST[0]}-{COHORT_LIST[-1]} ({len(COHORT_LIST)})  Threads for steps 6-7: {N_THREADS}")
print("Output:", OUT_DIR)
T_START = time.time()

In [0]:
# Same results as the numpy versions in steps 6-7 (checked against them in every test run), but each feature is sorted once
# and features run in parallel on all driver cores, so the full HCP universe (millions of rows) takes minutes, not hours.
import numba as nb
nb.set_num_threads(min(N_THREADS, nb.config.NUMBA_NUM_THREADS))

@nb.njit(parallel=True)
def nb_discrimination(Xc, ww, wn):
    """Per column: weighted mean, sd and median for writers / non-writers and the weighted AUC (ties share half the credit)."""
    n, p = Xc.shape
    Ww = ww.sum(); Wn = wn.sum()
    m_w = np.empty(p); m_n = np.empty(p); sd_w = np.empty(p); sd_n = np.empty(p)
    med_w = np.empty(p); med_n = np.empty(p); auc = np.empty(p)
    for j in nb.prange(p):
        x = Xc[:, j]
        s1w = 0.0; s2w = 0.0; s1n = 0.0; s2n = 0.0
        for i in range(n):
            v = x[i]
            s1w += ww[i] * v; s2w += ww[i] * v * v; s1n += wn[i] * v; s2n += wn[i] * v * v
        m_w[j] = s1w / Ww; m_n[j] = s1n / Wn
        sd_w[j] = np.sqrt(max(s2w / Ww - m_w[j] ** 2, 0.0)); sd_n[j] = np.sqrt(max(s2n / Wn - m_n[j] ** 2, 0.0))
        o = np.argsort(x, kind="mergesort")
        cw = 0.0; cn = 0.0; fw = False; fn = False
        below_n = 0.0; a = 0.0; gw = 0.0; gn = 0.0
        for k in range(n):
            i = o[k]; v = x[i]
            if k > 0 and v != x[o[k - 1]]:
                a += gw * (below_n + 0.5 * gn); below_n += gn; gw = 0.0; gn = 0.0
            gw += ww[i]; gn += wn[i]; cw += ww[i]; cn += wn[i]
            if not fw and cw >= Ww / 2:
                med_w[j] = v; fw = True
            if not fn and cn >= Wn / 2:
                med_n[j] = v; fn = True
        a += gw * (below_n + 0.5 * gn)
        if not fw: med_w[j] = x[o[0]]
        if not fn: med_n[j] = x[o[0]]
        auc[j] = min(max(a / (Ww * Wn), 0.0), 1.0)   # float rounding can give 1 + 1e-12 for a perfectly separating feature
    return m_w, m_n, sd_w, sd_n, med_w, med_n, auc

@nb.njit
def _corr_eta(R, v, y, w, nbins, m):
    """Rows where m: weighted correlation of the ranks R with v, and weighted correlation ratio of y over nbins rank bins."""
    W = 0.0; sR = 0.0; sv = 0.0; sy = 0.0; cnt = 0
    for i in range(R.shape[0]):
        if m[i]:
            W += w[i]; sR += w[i] * R[i]; sv += w[i] * v[i]; sy += w[i] * y[i]; cnt += 1
    mR = sR / W; mv = sv / W; ybar = sy / W
    num = 0.0; dR = 0.0; dv = 0.0; tot = 0.0
    wb = np.zeros(nbins); yb = np.zeros(nbins)
    for i in range(R.shape[0]):
        if m[i]:
            rc = R[i] - mR; vc = v[i] - mv
            num += w[i] * vc * rc; dR += w[i] * rc * rc; dv += w[i] * vc * vc
            tot += w[i] * (y[i] - ybar) ** 2
            b = min(int((R[i] - 1) / cnt * nbins), nbins - 1)
            wb[b] += w[i]; yb[b] += w[i] * y[i]
    den = np.sqrt(dR * dv)
    rho = num / den if den > 0 else 0.0
    e = 0.0
    for b in range(nbins):
        if wb[b] > 0:
            e += wb[b] * (yb[b] / wb[b] - ybar) ** 2
    eta = np.sqrt(e / tot) if tot > 0 else e
    return rho, eta

# columns returned by nb_group_stats, in order
GROUP_STAT_COLS = ["CONSTANT", "TOP_VALUE_SHARE", "RHO", "ETA", "RHO_NOISE", "ETA_NOISE", "RHO_HALF_A", "ETA_HALF_A", "RHO_HALF_B", "ETA_HALF_B"]

@nb.njit(parallel=True)
def nb_group_stats(Xg, w, y, ry, perm, rperm, half, ryA, ryB, nbins):
    """Per column of one group: constant, top value share, rank correlation (RHO) and correlation ratio (ETA) with NBRx,
    with shuffled NBRx (noise) and within each half of the doctors. ry / rperm / ryA / ryB are average ranks of the targets."""
    n, p = Xg.shape
    out = np.zeros((p, 10))
    mA = half == 0; mB = half == 1
    allm = np.ones(n, np.bool_)
    W = w.sum()
    for j in nb.prange(p):
        x = Xg[:, j]
        o = np.argsort(x, kind="mergesort")
        R = np.empty(n); RA = np.empty(n); RB = np.empty(n)
        k = 0; seenA = 0; seenB = 0; top = 0.0
        while k < n:   # one tie group at a time: average ranks overall and within each half
            e = k; gwt = 0.0; nA = 0; nB = 0
            while e < n and x[o[e]] == x[o[k]]:
                gwt += w[o[e]]
                if half[o[e]] == 0:
                    nA += 1
                else:
                    nB += 1
                e += 1
            r = (k + 1 + e) / 2.0
            rA = seenA + (nA + 1) / 2.0; rB = seenB + (nB + 1) / 2.0
            for q in range(k, e):
                i = o[q]; R[i] = r
                if half[i] == 0:
                    RA[i] = rA
                else:
                    RB[i] = rB
            seenA += nA; seenB += nB
            if gwt > top:
                top = gwt
            k = e
        out[j, 0] = 1.0 if x[o[0]] == x[o[n - 1]] else 0.0
        out[j, 1] = top / W
        out[j, 2], out[j, 3] = _corr_eta(R, ry, y, w, nbins, allm)
        out[j, 4], out[j, 5] = _corr_eta(R, rperm, perm, w, nbins, allm)
        out[j, 6], out[j, 7] = _corr_eta(RA, ryA, y, w, nbins, mA)
        out[j, 8], out[j, 9] = _corr_eta(RB, ryB, y, w, nbins, mB)
    return out

def pass_cols(n_rows, max_cols):
    """Features per pass so that one pass holds about 1e9 values (8 GB as float64)."""
    return max(10, min(max_cols, int(1e9 // max(n_rows, 1))))

# compile once on a tiny input
_x = np.asfortranarray(np.arange(12, dtype=np.float64).reshape(6, 2)); _w = np.ones(6); _h = np.array([0, 1, 0, 1, 0, 1])
nb_discrimination(_x, _w * (_h == 0), _w * (_h == 1))
nb_group_stats(_x, _w, _w, _w, _w, _w, _h, _w, _w, N_BINS)
print(f"numba {nb.__version__}: {nb.get_num_threads()} threads")

In [0]:
def read_source(t):
    """Raw source table with its cohort column renamed to COHORT_ID and HCP_COHORT_ID built if missing."""
    df = spark.read.format(t["fmt"]).load(t["path"])
    if t["cohort_col"] != "COHORT_ID":
        df = df.withColumnRenamed(t["cohort_col"], "COHORT_ID")
    if "HCP_COHORT_ID" not in df.columns:
        df = df.withColumn("HCP_COHORT_ID", F.concat(F.col("PROVIDER_ID").cast("string"), F.lit("_"), F.col("COHORT_ID")))
    return df

def feature_columns(t, columns):
    drop = {c.upper() for c in t["drop_cols"]} | METADATA_COLS | KEY_COLS
    return [c for c in columns if c.upper() not in drop]

FEATURES_BY_TABLE = {t["name"]: feature_columns(t, read_source(t).columns) for t in SOURCE_TABLES}
# every source table must hold every chosen cohort, written as in the cohort file (c1 ... c28); otherwise the cohort filter in step 4 would drop its rows
_missing_cohorts = {t["name"]: sorted(set(COHORT_LIST) - {r[0] for r in read_source(t).select(F.col("COHORT_ID").cast("string")).distinct().collect()})
                    for t in SOURCE_TABLES}
_missing_cohorts = {k: v for k, v in _missing_cohorts.items() if v}
assert not _missing_cohorts, f"source tables without some chosen cohorts: {_missing_cohorts}"
_owner, _dups = {}, []
for name, cols in FEATURES_BY_TABLE.items():
    for c in cols:
        if c.upper() in _owner:
            _dups.append((c, _owner[c.upper()], name))
        _owner[c.upper()] = name
assert not _dups, f"feature names in two tables: {_dups[:20]}"
N_FEATURES_ALL = sum(len(v) for v in FEATURES_BY_TABLE.values())
if MAX_FEATURES:   # test mode: features spread evenly over the tables
    _all = [(n, c) for n, cols in FEATURES_BY_TABLE.items() for c in cols]
    _keep = set(_all[::max(1, len(_all) // MAX_FEATURES)][:MAX_FEATURES])
    FEATURES_BY_TABLE = {n: [c for c in cols if (n, c) in _keep] for n, cols in FEATURES_BY_TABLE.items()}

# Fill rules (Combine D14 / D10): 0 for counts and flags; -9999 first occurrence; 9999 days / months since;
# median of the training cohorts c1-c24 for census features and HCP_EXPERIENCE
FIRST_OCC_FILL  = -9999.0
TIME_SINCE_FILL = 9999.0
RE_TIME_SINCE   = re.compile(r"(DAYS_SINCE|MONTHS_SINCE|TIME_TO_ADOPT|MIN_DAYS_TO_)")
MEDIAN_TABLES   = {"census_features"}
MEDIAN_COLUMNS  = {"HCP_EXPERIENCE"}

def impute_rule(table, col):
    if col.startswith("MAX_EVENT_TIME_") and col.endswith("_FIRST_OCCURRENCE"):
        return "first_occurrence"
    if RE_TIME_SINCE.search(col):
        return "time_since"
    if table in MEDIAN_TABLES or col in MEDIAN_COLUMNS:
        return "median"
    return "zero"

RULES = {(name, c): impute_rule(name, c) for name, cols in FEATURES_BY_TABLE.items() for c in cols}
MEDIANS = {}
for name in {n for (n, _), r in RULES.items() if r == "median"}:   # full universe, c1-c24, as in Combine
    t = next(t for t in SOURCE_TABLES if t["name"] == name)
    cols = [c for (n, c), r in RULES.items() if n == name and r == "median"]
    train = read_source(t).filter(F.col("COHORT_ID").isin(TRAINING_COHORTS) & F.col("PROVIDER_ID").isNotNull())
    row = train.agg(*[F.percentile_approx(F.col(f"`{c}`").cast("double"), 0.5).alias(c) for c in cols]).first()
    MEDIANS.update({c: (float(row[c]) if row[c] is not None else 0.0) for c in cols})

def fill_value(table, col):
    r = RULES[(table, col)]
    return {"first_occurrence": FIRST_OCC_FILL, "time_since": TIME_SINCE_FILL, "zero": 0.0}.get(r, MEDIANS.get(col, 0.0))

CALENDAR = ["CALENDAR_MONTH", "CALENDAR_YEAR", "MONTHS_SINCE_KER_LAUNCH"] if ADD_CALENDAR else []
dictionary = pd.DataFrame(
    [(c, t["name"], t["sub_category"], RULES[(t["name"], c)], float(fill_value(t["name"], c)))
     for t in SOURCE_TABLES for c in FEATURES_BY_TABLE[t["name"]]]
    + [(c, "derived", "Calendar", "none", np.nan) for c in CALENDAR],
    columns=["FEATURE", "SOURCE_TABLE", "SUB_CATEGORY", "IMPUTE_RULE", "IMPUTE_VALUE"]).set_index("FEATURE")
assert dictionary.index.is_unique
FEATURES = list(dictionary.index)
print(f"{len(FEATURES):,} features used (of {N_FEATURES_ALL + len(CALENDAR):,}) from {len(SOURCE_TABLES)} tables; "
      f"fill rules: {dictionary.IMPUTE_RULE.value_counts().to_dict()}")
display(dictionary.reset_index().groupby(["SOURCE_TABLE", "SUB_CATEGORY"]).FEATURE.count().reset_index(name="features"))

In [0]:
cohorts = spark.read.parquet(COHORTS_PATH).select(F.col("cohort_id").alias("COHORT_ID"), F.col("dataset").alias("DATASET"),
                                                  F.col("lookback_end_date").cast("date").alias("LOOKBACK_END_DATE"))
assert cohorts.count() == 28, "cohort file does not hold c1-c28"
cohorts = cohorts.filter(F.col("COHORT_ID").isin(COHORT_LIST))   # COHORTS widget
assert cohorts.count() == len(COHORT_LIST), "a chosen cohort is not in the cohort file"
universe = (spark.read.parquet(UNIVERSE_PATH).select(F.col("PROVIDER_ID").cast("string").alias("PROVIDER_ID")).distinct()
            .filter(F.col("PROVIDER_ID").isNotNull() & (F.col("PROVIDER_ID") != "0")))

# HCPs: the sample list's HCPs with their sample group and group weight (each HCP is used in every chosen cohort)
if SAMPLE_LIST:
    hp = (spark.read.format("delta").load(SAMPLE_LIST)
          .select(F.col("PROVIDER_ID").cast("string").alias("PROVIDER_ID"), "SAMPLE_GROUP", "WEIGHT_GROUP").distinct()
          .join(universe, "PROVIDER_ID", "inner").toPandas())
else:
    hp = universe.toPandas().assign(SAMPLE_GROUP="all", WEIGHT_GROUP=1.0)
assert hp.PROVIDER_ID.is_unique, "an HCP has two sample groups or weights"
if TEST_HCPS:   # same group mix as the sample, fixed by a hash of the HCP id
    hp["_h"] = pd.util.hash_pandas_object(hp.PROVIDER_ID, index=False).values
    hp["_r"] = hp.groupby("SAMPLE_GROUP")._h.rank(method="first")
    hp["_n"] = hp.groupby("SAMPLE_GROUP").PROVIDER_ID.transform("size")
    hp = hp[hp._r <= np.maximum(1, np.round(hp._n * TEST_HCPS / len(hp)))].drop(columns=["_h", "_r", "_n"])

target = spark.read.format("delta").load(TARGET_PATH).select("HCP_COHORT_ID", F.col("KERENDIA_NBRX").cast("double").alias("TARGET_NBRX"))
writer = spark.read.format("delta").load(WRITER_PATH).select("HCP_COHORT_ID", F.col(WRITER_COL).cast("double").alias("WRITER_NBRX_12M"))
skeleton = (spark.createDataFrame(hp[["PROVIDER_ID", "SAMPLE_GROUP", "WEIGHT_GROUP"]])
            .crossJoin(F.broadcast(cohorts))
            .withColumn("HCP_COHORT_ID", F.concat_ws("_", "PROVIDER_ID", "COHORT_ID"))
            .join(target, "HCP_COHORT_ID", "left")
            .join(writer, "HCP_COHORT_ID", "left")
            # Step 1, writer definition: Kerendia NBRx in the last 12 months > 0 (empty = 0)
            .withColumn("WRITER_NBRX_12M", F.coalesce("WRITER_NBRX_12M", F.lit(0.0)))
            .withColumn("IS_WRITER", F.col("WRITER_NBRX_12M") > 0))
skeleton.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(KEYS_PATH)

keys = spark.read.format("delta").load(KEYS_PATH).toPandas().sort_values("HCP_COHORT_ID").reset_index(drop=True)
assert len(keys) == len(hp) * len(COHORT_LIST) and keys.HCP_COHORT_ID.is_unique, "rows are not one per HCP x cohort"
assert keys.TARGET_NBRX.notna().all(), "rows without a target value"
keys["COHORT_N"] = keys.COHORT_ID.str[1:].astype(int)
keys["WEIGHT"]   = keys.WEIGHT_GROUP.astype(float)   # every chosen month is used, so only the group weight applies (1 for the full universe)
keys["HALF"]     = (pd.util.hash_pandas_object(keys.PROVIDER_ID, index=False).values % 2).astype(int)   # flag 6

# Step 2: the two modeling data sets (row positions into keys / X)
IDX = {g: np.where(keys.IS_WRITER.values == flag)[0] for g, flag in GROUPS.items()}
group_summary = pd.DataFrame([
    (g, len(i), keys.iloc[i].PROVIDER_ID.nunique(), float(keys.WEIGHT.values[i].sum()),
     float(keys.TARGET_NBRX.values[i].sum()), int((keys.TARGET_NBRX.values[i] > 0).sum()), keys.iloc[i].COHORT_ID.nunique())
    for g, i in IDX.items()], columns=["group", "rows", "hcps", "weighted_rows", "nbrx", "rows_with_nbrx", "cohorts"])
assert group_summary.rows.sum() == len(keys)
print(f"{len(hp):,} HCPs x {len(COHORT_LIST)} cohorts ({COHORT_LIST[0]}-{COHORT_LIST[-1]}) = {len(keys):,} rows")
display(group_summary)

In [0]:
t0 = time.time()
keys_sdf = spark.read.format("delta").load(KEYS_PATH).select("HCP_COHORT_ID", "LOOKBACK_END_DATE")
kb = keys_sdf.select("HCP_COHORT_ID")
if len(keys) <= 1_000_000:   # small key lists are broadcast; for the full universe Spark picks the join itself
    kb = F.broadcast(kb)
out = keys_sdf
for t in SOURCE_TABLES:
    cols = FEATURES_BY_TABLE[t["name"]]
    if not cols:
        continue
    src = (read_source(t)
           .filter(F.col("PROVIDER_ID").isNotNull() & (F.col("PROVIDER_ID").cast("string") != "0")
                   & F.col("COHORT_ID").isin(COHORT_LIST))
           .select("HCP_COHORT_ID", *[F.col(f"`{c}`") for c in cols])
           .join(kb, "HCP_COHORT_ID", "left_semi"))   # only the sample rows are joined
    out = out.join(src, "HCP_COHORT_ID", "left")

exprs = [F.coalesce(F.col(f"`{c}`").cast("double"), F.lit(float(dictionary.IMPUTE_VALUE[c]))).alias(c)
         for c in FEATURES if c not in CALENDAR]
if ADD_CALENDAR:   # same as Combine cell 9
    exprs += [F.month("LOOKBACK_END_DATE").cast("double").alias("CALENDAR_MONTH"),
              F.year("LOOKBACK_END_DATE").cast("double").alias("CALENDAR_YEAR"),
              F.floor(F.months_between("LOOKBACK_END_DATE", F.lit("2021-07-01").cast("date"))).cast("double").alias("MONTHS_SINCE_KER_LAUNCH")]
out.select("HCP_COHORT_ID", *exprs).write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(DATA_PATH)

data = spark.read.format("delta").load(DATA_PATH)
_c = data.agg(F.count("*").alias("rows"), F.countDistinct("HCP_COHORT_ID").alias("keys")).first()
assert _c["rows"] == _c["keys"] == len(keys), f"category data has {_c['rows']:,} rows / {_c['keys']:,} keys, expected {len(keys):,} (duplicate source keys?)"
print(f"Category data written: {_c['rows']:,} rows x {len(FEATURES):,} features ({(time.time() - t0) / 60:.1f} min)")

In [0]:
t0 = time.time()
X = np.empty((len(keys), len(FEATURES)), dtype=np.float32)
LOAD_COLS = max(20, min(1000, int(2.5e8 // len(keys))))   # about 2 GB per pass, whatever the row count
for s in range(0, len(FEATURES), LOAD_COLS):
    cols = FEATURES[s:s + LOAD_COLS]
    pdf = (data.select("HCP_COHORT_ID", *[F.col(f"`{c}`") for c in cols]).toPandas()
           .set_index("HCP_COHORT_ID").reindex(keys.HCP_COHORT_ID))
    X[:, s:s + len(cols)] = pdf[cols].to_numpy(dtype=np.float32)
    del pdf
assert not np.isnan(X).any(), "empty values or unmatched rows after the fill"
FILL = dictionary.IMPUTE_VALUE.reindex(FEATURES).to_numpy(dtype=np.float32)   # NaN = no fill rule (always counts as filled)
W_ALL, Y_ALL, H_ALL, ISW = keys.WEIGHT.values, keys.TARGET_NBRX.values, keys.HALF.values, keys.IS_WRITER.values
print(f"X: {X.shape[0]:,} x {X.shape[1]:,} ({X.nbytes / 1e9:.1f} GB, {(time.time() - t0) / 60:.1f} min)")

In [0]:
def discrimination_stats(Xc, isw, w):
    """numpy reference for nb_discrimination (used in test runs to check it). Weighted mean, sd, median per group and
    weighted AUC (chance a writer row has a higher value than a non-writer row)."""
    n, p = Xc.shape
    ww, wn = w * isw, w * ~isw
    Ww, Wn = ww.sum(), wn.sum()
    m_w, m_n = (ww @ Xc) / Ww, (wn @ Xc) / Wn
    sd_w = np.sqrt(np.maximum((ww @ Xc ** 2) / Ww - m_w ** 2, 0))
    sd_n = np.sqrt(np.maximum((wn @ Xc ** 2) / Wn - m_n ** 2, 0))
    order = np.argsort(Xc, axis=0, kind="stable")
    xs = np.take_along_axis(Xc, order, 0)
    sw, sn = ww[order], wn[order]
    cols = np.arange(p)
    med_w = xs[(np.cumsum(sw, 0) >= Ww / 2).argmax(0), cols]
    med_n = xs[(np.cumsum(sn, 0) >= Wn / 2).argmax(0), cols]
    # ties share half the credit: AUC = sum over tied values of W_writer x (W_non_writer below + W_non_writer tied / 2)
    new = np.ones((n, p), bool); new[1:] = xs[1:] != xs[:-1]
    gid = (np.cumsum(new, 0) - 1) + cols * n
    gw = np.bincount(gid.ravel(), sw.ravel(), minlength=n * p).reshape(p, n)
    gn = np.bincount(gid.ravel(), sn.ravel(), minlength=n * p).reshape(p, n)
    auc = (gw * (np.cumsum(gn, 1) - gn + 0.5 * gn)).sum(1) / (Ww * Wn)
    auc = np.clip(auc, 0.0, 1.0)   # float rounding can give 1 + 1e-12 for a perfectly separating feature
    return m_w, m_n, sd_w, sd_n, med_w, med_n, auc

t0 = time.time()
parts = []
ww_, wn_ = W_ALL * ISW, W_ALL * ~ISW
DISC_MAX_DIFF = 0.0
step = pass_cols(len(keys), DISC_CHUNK)
for s in range(0, len(FEATURES), step):
    cols = FEATURES[s:s + step]
    Xc = np.asfortranarray(X[:, s:s + len(cols)], dtype=np.float64)
    m_w, m_n, sd_w, sd_n, med_w, med_n, auc = nb_discrimination(Xc, ww_, wn_)
    if TEST:   # numba = numpy reference
        ref = discrimination_stats(Xc, ISW, W_ALL)
        for a, b in zip((m_w, m_n, med_w, med_n, auc), (ref[0], ref[1], ref[4], ref[5], ref[6])):
            DISC_MAX_DIFF = max(DISC_MAX_DIFF, float(np.max(np.abs(a - b) / np.maximum(1.0, np.abs(b)))))
        for a, b, m in [(sd_w, ref[2], ref[0]), (sd_n, ref[3], ref[1])]:   # variance = E[x^2] - mean^2: compare on the mean^2 scale
            DISC_MAX_DIFF = max(DISC_MAX_DIFF, float(np.max(np.abs(a ** 2 - b ** 2) / np.maximum(1.0, m ** 2))))
    filled = (Xc != FILL[s:s + len(cols)])
    parts.append(pd.DataFrame({
        "FEATURE": cols, "MEAN_WRITERS": m_w, "MEAN_NON_WRITERS": m_n, "MEDIAN_WRITERS": med_w, "MEDIAN_NON_WRITERS": med_n,
        "SD_WRITERS": sd_w, "SD_NON_WRITERS": sd_n,
        "FILL_SHARE_WRITERS": (ww_ @ filled) / ww_.sum(), "FILL_SHARE_NON_WRITERS": (wn_ @ filled) / wn_.sum(), "AUC": auc}))
    del Xc, filled
disc = pd.concat(parts, ignore_index=True)
if TEST:
    assert DISC_MAX_DIFF < 1e-9, f"numba mean / median analysis differs from numpy: {DISC_MAX_DIFF:.2e}"
    print(f"numba = numpy check passed (max relative difference {DISC_MAX_DIFF:.1e})")
pooled = np.sqrt((disc.SD_WRITERS ** 2 + disc.SD_NON_WRITERS ** 2) / 2)
disc["MEAN_DIFF"]   = disc.MEAN_WRITERS - disc.MEAN_NON_WRITERS
disc["MEDIAN_DIFF"] = disc.MEDIAN_WRITERS - disc.MEDIAN_NON_WRITERS
disc["STD_MEAN_DIFF"] = np.where(pooled > 0, disc.MEAN_DIFF / pooled.where(pooled > 0, 1), 0.0)   # Cohen's d
disc["DISCRIMINATION"] = (2 * disc.AUC - 1).abs()   # 0 = same values in both groups, 1 = no overlap
disc["HIGHER_IN"] = np.select([disc.AUC > 0.5, disc.AUC < 0.5], ["writers", "non_writers"], "same")
disc["SUB_CATEGORY"] = dictionary.SUB_CATEGORY.reindex(disc.FEATURE).values
disc = disc.sort_values(["DISCRIMINATION", "STD_MEAN_DIFF"], ascending=False, key=lambda s: s.abs()).reset_index(drop=True)
disc["DISCRIMINATION_RANK"] = np.arange(1, len(disc) + 1)
assert disc.FEATURE.is_unique and len(disc) == len(FEATURES) and disc.AUC.between(0, 1).all()
print(f"Mean / median analysis of {len(disc):,} features ({(time.time() - t0) / 60:.1f} min). "
      f"Features with DISCRIMINATION >= 0.2: {int((disc.DISCRIMINATION >= 0.2).sum()):,}; >= 0.5: {int((disc.DISCRIMINATION >= 0.5).sum()):,}")
print("Means include the fill values (-9999 / 9999); the medians and AUC are not distorted by them.")
display(disc.head(TOP_N)[["DISCRIMINATION_RANK", "FEATURE", "SUB_CATEGORY", "DISCRIMINATION", "HIGHER_IN", "AUC", "STD_MEAN_DIFF",
                          "MEAN_WRITERS", "MEAN_NON_WRITERS", "MEDIAN_WRITERS", "MEDIAN_NON_WRITERS",
                          "FILL_SHARE_WRITERS", "FILL_SHARE_NON_WRITERS"]])
display(disc.groupby("SUB_CATEGORY").agg(features=("FEATURE", "size"), max_discrimination=("DISCRIMINATION", "max"),
                                         median_discrimination=("DISCRIMINATION", "median"),
                                         features_ge_0_2=("DISCRIMINATION", lambda s: int((s >= 0.2).sum())))
        .sort_values("max_discrimination", ascending=False).reset_index())

In [0]:
def wcorr(R, v, w):
    """Weighted Pearson correlation of each column of R with v (on ranks = weighted Spearman)."""
    W = w.sum()
    Rc = R - (w @ R) / W
    vc = v - (w @ v) / W
    with np.errstate(invalid="ignore", divide="ignore"):
        r = ((w * vc) @ Rc) / np.sqrt((w @ (Rc ** 2)) * (w @ (vc ** 2)))
    return np.nan_to_num(r)

def weta(R, y, w):
    """Weighted correlation ratio: how much NBRx differs between 10 rank bins of the feature (catches non-linear links)."""
    n, p = R.shape
    bins = np.minimum(((R - 1) / n * N_BINS).astype(int), N_BINS - 1)
    W = w.sum(); ybar = (w @ y) / W
    tot = w @ (y - ybar) ** 2
    out = np.zeros(p)
    for j in range(p):
        wb = np.bincount(bins[:, j], weights=w, minlength=N_BINS)
        yb = np.bincount(bins[:, j], weights=w * y, minlength=N_BINS)
        ok = wb > 0
        out[j] = (wb[ok] * (yb[ok] / wb[ok] - ybar) ** 2).sum()
    return np.sqrt(out / tot) if tot > 0 else out

def top_share(X, w):
    """Weighted share of rows holding each column's most common value."""
    out = np.empty(X.shape[1])
    for j in range(X.shape[1]):
        _, inv = np.unique(X[:, j], return_inverse=True)
        out[j] = np.bincount(inv, weights=w).max() / w.sum()
    return out

def doctor_block_permutation(df, rng):
    """NBRx shuffled between doctors with the same number of rows; each doctor's months stay together."""
    d = pd.DataFrame({"pos": np.arange(len(df)), "P": df.PROVIDER_ID.values, "k": df.COHORT_N.values, "y": df.TARGET_NBRX.values})
    d["n"] = d.groupby("P").P.transform("size")
    d = d.sort_values(["n", "P", "k"])
    out = np.empty(len(d))
    for n, sub in d.groupby("n", sort=False):
        y = sub.y.values.reshape(-1, n)
        out[sub.pos.values] = y[rng.permutation(len(y))].ravel()
    return out

rng = np.random.default_rng(SEED)
SUB, PERM = {}, {}
for g, idx in IDX.items():
    k = keys.iloc[idx]
    SUB[g] = np.sort(rng.choice(len(idx), size=min(COPY_SAMPLE_ROWS, len(idx)), replace=False))
    PERM[g] = doctor_block_permutation(k, rng)
    assert np.isclose(PERM[g].sum(), k.TARGET_NBRX.sum()), "shuffle changed the NBRx total"

stats_parts = {g: [] for g in GROUPS}
copy_ranks  = {g: [] for g in GROUPS}
copy_names  = {g: [] for g in GROUPS}
def numpy_group_stats(Xg, w, y, perm, half):
    """numpy reference for nb_group_stats (used in test runs to check it)."""
    R = rankdata(Xg, axis=0)
    st = {"CONSTANT": (Xg == Xg[0]).all(axis=0), "TOP_VALUE_SHARE": top_share(Xg, w),
          "RHO": wcorr(R, rankdata(y), w), "ETA": weta(R, y, w),
          "RHO_NOISE": wcorr(R, rankdata(perm), w), "ETA_NOISE": weta(R, perm, w)}
    for h, lab in [(0, "A"), (1, "B")]:
        m = half == h
        Rh = rankdata(Xg[m], axis=0)
        st[f"RHO_HALF_{lab}"] = wcorr(Rh, rankdata(y[m]), w[m])
        st[f"ETA_HALF_{lab}"] = weta(Rh, y[m], w[m])
    return pd.DataFrame(st)

TARGET_RANKS = {}   # average ranks of NBRx, of the shuffled NBRx and of NBRx within each half of the doctors
for g, idx in IDX.items():
    y, half = Y_ALL[idx], H_ALL[idx]
    ryA, ryB = np.zeros(len(idx)), np.zeros(len(idx))
    ryA[half == 0], ryB[half == 1] = rankdata(y[half == 0]), rankdata(y[half == 1])
    TARGET_RANKS[g] = (rankdata(y), rankdata(PERM[g]), ryA, ryB)

STATS_MAX_DIFF = 0.0
t0 = time.time()
step = pass_cols(max(len(i) for i in IDX.values()), CHUNK)
for s in range(0, len(FEATURES), step):
    cols = FEATURES[s:s + step]
    fill = FILL[s:s + len(cols)]
    for g, idx in IDX.items():
        Xg = np.asfortranarray(X[idx, s:s + len(cols)], dtype=np.float64)
        w, y, half = W_ALL[idx], Y_ALL[idx], H_ALL[idx].astype(np.int64)
        ry, rperm, ryA, ryB = TARGET_RANKS[g]
        st = pd.DataFrame(nb_group_stats(Xg, w, y, ry, PERM[g], rperm, half, ryA, ryB, N_BINS), columns=GROUP_STAT_COLS)
        st["CONSTANT"] = st.CONSTANT > 0.5
        st.insert(0, "FEATURE", cols)
        st.insert(3, "FILL_SHARE", (w @ (Xg != fill).astype(np.float64)) / w.sum())
        constant = st.CONSTANT.values
        if TEST:   # numba = numpy reference
            ref = numpy_group_stats(Xg, w, y, PERM[g], half)
            assert (ref.CONSTANT.values == constant).all(), "numba and numpy disagree on constant features"
            num = [c for c in GROUP_STAT_COLS if c != "CONSTANT"]
            STATS_MAX_DIFF = max(STATS_MAX_DIFF, float(np.max(np.abs(st[num].values - ref[num].values))))
        stats_parts[g].append(st)
        keep = (~constant) & (st.TOP_VALUE_SHARE.values < NEAR_CONSTANT_SHARE) & (st.FILL_SHARE.values >= MIN_FILL_SHARE)
        if keep.any():
            copy_ranks[g].append(rankdata(Xg[SUB[g]][:, keep], axis=0).astype(np.float32))
            copy_names[g] += [c for c, k in zip(cols, keep) if k]
        del Xg
    print(f"features {s + len(cols):,}/{len(FEATURES):,}  ({(time.time() - t0) / 60:.1f} min)")
print({g: len(v) for g, v in copy_names.items()}, "features reach stage 4")
if TEST:
    assert STATS_MAX_DIFF < 1e-6, f"numba statistics differ from numpy: {STATS_MAX_DIFF:.2e}"   # ETA = sqrt(.) magnifies rounding near 0
    print(f"numba = numpy check passed (max difference {STATS_MAX_DIFF:.1e})")

In [0]:
RESULTS, THRESHOLDS, KEPT_MAX_COPY_RHO = {}, {}, {}
for g in GROUPS:
    st = pd.concat(stats_parts[g], ignore_index=True)
    st["SUB_CATEGORY"] = dictionary.SUB_CATEGORY.reindex(st.FEATURE).values
    live = ~st.CONSTANT
    thr_rho = float(np.quantile(st.RHO_NOISE[live].abs(), NOISE_QUANTILE)) if live.any() else 0.0
    thr_eta = float(np.quantile(st.ETA_NOISE[live], NOISE_QUANTILE)) if live.any() else 0.0
    THRESHOLDS[g] = {"rho_noise_bar": thr_rho, "eta_noise_bar": thr_eta}
    st["SIGNAL_LINEAR"] = st.RHO.abs() >= thr_rho
    st["SIGNAL_NONLINEAR"] = st.ETA >= thr_eta
    st["SIGNAL_SCORE"] = np.maximum(st.RHO.abs() / thr_rho, st.ETA / thr_eta)   # 1 = at the noise bar
    st["REMOVED_AT"] = None

    def drop(mask, stage):
        m = np.asarray(mask) & st.REMOVED_AT.isna().values
        st.loc[m, "REMOVED_AT"] = stage

    drop(st.CONSTANT, "1_constant")
    drop(st.TOP_VALUE_SHARE >= NEAR_CONSTANT_SHARE, "2_near_constant")
    drop(st.FILL_SHARE < MIN_FILL_SHARE, "3_low_fill")

    # Stage 4: strongest feature first; a feature is dropped if it is a near-copy of one already kept
    names = copy_names[g]
    assert set(names) == set(st.FEATURE[st.REMOVED_AT.isna()]), "stage 4 candidates do not match stages 1-3"
    copy_of = {}
    if names:
        Z = np.hstack(copy_ranks[g])
        sd = Z.std(axis=0)
        Z = np.where(sd > 0, (Z - Z.mean(axis=0)) / np.where(sd > 0, sd, 1), 0).astype(np.float32)
        C = (Z.T @ Z) / len(Z)
        del Z
        sc = st.set_index("FEATURE").loc[names]
        order = np.lexsort((-sc.FILL_SHARE.values, -sc.SIGNAL_SCORE.values))
        kept = []
        for i in order:
            if kept:
                c = np.abs(C[i, kept]); j = int(c.argmax())
                if c[j] >= NEAR_COPY_RHO:
                    copy_of[names[i]] = (names[kept[j]], float(C[i, kept[j]]))
                    continue
            kept.append(i)
    st["COPY_OF"] = st.FEATURE.map(lambda f: copy_of.get(f, (None, None))[0])
    st["RHO_WITH_COPY"] = st.FEATURE.map(lambda f: copy_of.get(f, (None, None))[1])
    drop(st.FEATURE.isin(list(copy_of)), "4_near_copy")

    # Flags 5 and 6 only mark features; they stay in the model
    st["SELECTED"] = st.REMOVED_AT.isna()
    st["REASON"] = st.REMOVED_AT.map(STAGES)
    no_signal = ~st.SIGNAL_LINEAR & ~st.SIGNAL_NONLINEAR
    unstable = np.where(st.SIGNAL_LINEAR, np.sign(st.RHO_HALF_A) != np.sign(st.RHO_HALF_B),
                        np.minimum(st.ETA_HALF_A, st.ETA_HALF_B) < thr_eta)
    st["FLAG_5_NO_SIGNAL"] = st.SELECTED & no_signal
    st["FLAG_6_UNSTABLE"] = st.SELECTED & ~no_signal & unstable
    st["PASSES_ALL_SIX"] = st.SELECTED & ~st.FLAG_5_NO_SIGNAL & ~st.FLAG_6_UNSTABLE

    sel_idx = [names.index(f) for f in st.FEATURE[st.SELECTED]]
    if len(sel_idx) > 1:
        Cs = np.abs(C[np.ix_(sel_idx, sel_idx)]); np.fill_diagonal(Cs, 0)
        KEPT_MAX_COPY_RHO[g] = float(Cs.max())
        del Cs
    else:
        KEPT_MAX_COPY_RHO[g] = 0.0
    if names:
        del C
    RESULTS[g] = st
    print(f"{g}: noise bars rho={thr_rho:.4f} eta={thr_eta:.4f}; kept after stage 4: {int(st.SELECTED.sum()):,} of {len(st):,}; "
          f"flagged no signal {int(st.FLAG_5_NO_SIGNAL.sum()):,}, unstable {int(st.FLAG_6_UNSTABLE.sum()):,}; no flag {int(st.PASSES_ALL_SIX.sum()):,}")

funnel = []
for g, st in RESULTS.items():
    left = len(st)
    funnel.append((g, "0_start", "all candidate features", 0, left, None))
    for s_, d in STAGES.items():
        n = int((st.REMOVED_AT == s_).sum()); left -= n
        funnel.append((g, s_, d, n, left, None))
    for f_, d in FLAGS.items():
        funnel.append((g, f_, d, 0, left, int(st[f_].sum())))
    funnel.append((g, "no_flag", "kept and not flagged", 0, left, int(st.PASSES_ALL_SIX.sum())))
funnel = pd.DataFrame(funnel, columns=["group", "stage", "rule", "removed", "features_left", "flagged"])
display(funnel)

In [0]:
PARAMS = dict(objective="poisson", learning_rate=LEARNING_RATE, num_leaves=NUM_LEAVES, min_data_in_leaf=MIN_DATA_IN_LEAF,
              feature_fraction=1.0, bagging_fraction=1.0, bagging_freq=0, lambda_l1=0.0, lambda_l2=0.0,
              seed=SEED, deterministic=True, force_col_wise=True, num_threads=os.cpu_count(), verbose=-1)
COL_POS = {f: i for i, f in enumerate(FEATURES)}

def poisson_deviance(y, mu, w):
    with np.errstate(divide="ignore", invalid="ignore"):
        t = np.where(y > 0, y * np.log(y / mu), 0.0) - (y - mu)
    return 2 * float((w * t).sum())

MODELS, fit_rows = {}, []   # one model per group on every row (all chosen cohorts)
for g, idx in IDX.items():
    feats = RESULTS[g].FEATURE[RESULTS[g].SELECTED].tolist()
    cidx = np.array([COL_POS[f] for f in feats], dtype=int)
    y, w = Y_ALL[idx], W_ALL[idx]
    Xg = X[np.ix_(idx, cidx)]
    t0 = time.time()
    ds = lgb.Dataset(Xg, label=y, weight=w, feature_name=[f"f{i}" for i in range(len(feats))], free_raw_data=True)
    bst = lgb.train(PARAMS, ds, num_boost_round=N_TREES)
    mu = bst.predict(Xg)
    dev, dev0 = poisson_deviance(y, mu, w), poisson_deviance(y, np.full_like(y, (w @ y) / w.sum()), w)
    MODELS[g] = dict(booster=bst, features=feats, cidx=cidx)
    fit_rows.append((g, len(idx), len(feats), N_TREES, float(w @ y), float(w @ mu), 1 - dev / dev0,
                     float(pd.Series(mu).corr(pd.Series(y), method="spearman")), (time.time() - t0) / 60))
    del Xg, ds
fit_summary = pd.DataFrame(fit_rows, columns=["group", "rows", "features", "trees", "weighted_nbrx", "weighted_predicted_nbrx",
                                              "in_sample_deviance_explained", "in_sample_spearman", "minutes"])
print("In-sample fit only (the model is over-fitted on purpose to rank features; it is not a performance estimate).")
display(fit_summary)

In [0]:
def col_spearman(A, B):
    """Spearman correlation between matching columns of A and B."""
    Ra, Rb = rankdata(A, axis=0), rankdata(B, axis=0)
    Ra -= Ra.mean(0); Rb -= Rb.mean(0)
    with np.errstate(invalid="ignore", divide="ignore"):
        r = (Ra * Rb).sum(0) / np.sqrt((Ra ** 2).sum(0) * (Rb ** 2).sum(0))
    return np.nan_to_num(r)

SHAP, SHAP_MAX_ERR = {}, {}
for g, m in MODELS.items():
    idx, bst, p = IDX[g], m["booster"], len(m["features"])
    Xg, w = X[np.ix_(idx, m["cidx"])], W_ALL[idx]
    abs_sum, sum_, err = np.zeros(p), np.zeros(p), 0.0
    t0 = time.time()
    for s in range(0, len(idx), SHAP_CHUNK):
        c = bst.predict(Xg[s:s + SHAP_CHUNK], pred_contrib=True)          # p feature columns + 1 base value (log scale)
        raw = bst.predict(Xg[s:s + SHAP_CHUNK], raw_score=True)
        err = max(err, float(np.abs(c.sum(1) - raw).max()))                 # SHAP values add up to the prediction
        ws = w[s:s + SHAP_CHUNK]
        abs_sum += ws @ np.abs(c[:, :p]); sum_ += ws @ c[:, :p]
        del c
    sub = np.sort(np.random.default_rng(SEED).choice(len(idx), size=min(SHAP_DIRECTION_ROWS, len(idx)), replace=False))
    direction = col_spearman(Xg[sub].astype(np.float64), bst.predict(Xg[sub], pred_contrib=True)[:, :p])
    sh = pd.DataFrame({"FEATURE": m["features"], "MEAN_ABS_SHAP": abs_sum / w.sum(), "MEAN_SHAP": sum_ / w.sum(),
                       "SHAP_DIRECTION_RHO": direction,   # > 0: higher value -> more predicted NBRx
                       "GAIN": bst.feature_importance("gain"), "SPLITS": bst.feature_importance("split")})
    sh = sh.sort_values(["MEAN_ABS_SHAP", "GAIN"], ascending=False).reset_index(drop=True)
    sh["SHAP_RANK"] = np.arange(1, len(sh) + 1)
    tot = sh.MEAN_ABS_SHAP.sum()
    sh["SHAP_SHARE"] = sh.MEAN_ABS_SHAP / tot if tot > 0 else 0.0
    sh["SHAP_CUM_SHARE"] = sh.SHAP_SHARE.cumsum()
    SHAP[g], SHAP_MAX_ERR[g] = sh, err
    print(f"{g}: SHAP on {len(idx):,} rows x {p:,} features in {(time.time() - t0) / 60:.1f} min; "
          f"max |sum of SHAP - prediction| = {err:.2e}")
    del Xg

In [0]:
REPORT_COLS = ["SHAP_RANK", "FEATURE", "SUB_CATEGORY", "MEAN_ABS_SHAP", "SHAP_SHARE", "SHAP_CUM_SHARE", "SHAP_DIRECTION_RHO", "GAIN",
               "DISCRIMINATION_RANK", "DISCRIMINATION", "HIGHER_IN", "MEAN_WRITERS", "MEAN_NON_WRITERS", "MEDIAN_WRITERS", "MEDIAN_NON_WRITERS",
               "SIGNAL_SCORE", "RHO", "ETA", "FLAG_5_NO_SIGNAL", "FLAG_6_UNSTABLE", "FILL_SHARE"]
REPORT, coverage = {}, []
for g in GROUPS:
    r = (SHAP[g]
         .merge(RESULTS[g][["FEATURE", "SUB_CATEGORY", "SIGNAL_SCORE", "RHO", "ETA", "FLAG_5_NO_SIGNAL", "FLAG_6_UNSTABLE", "FILL_SHARE"]], on="FEATURE")
         .merge(disc[["FEATURE", "DISCRIMINATION_RANK", "DISCRIMINATION", "HIGHER_IN", "MEAN_WRITERS", "MEAN_NON_WRITERS",
                      "MEDIAN_WRITERS", "MEDIAN_NON_WRITERS"]], on="FEATURE"))
    assert len(r) == len(SHAP[g])
    REPORT[g] = r[REPORT_COLS]
    used = r[r.MEAN_ABS_SHAP > 0]
    coverage.append((g, len(r), len(used), *[int((r.SHAP_CUM_SHARE < q).sum()) + 1 if len(used) else 0 for q in (0.8, 0.9, 0.95)]))
coverage = pd.DataFrame(coverage, columns=["group", "features_in_model", "features_used_by_trees",
                                           "features_for_80pct_shap", "features_for_90pct_shap", "features_for_95pct_shap"])
display(coverage)

for g in GROUPS:
    print(f"Top {TOP_N} {CATEGORY} features for {g} (by mean |SHAP|)")
    display(REPORT[g].head(TOP_N))

sub_summary = (pd.concat([REPORT[g].assign(group=g) for g in GROUPS])
               .groupby(["group", "SUB_CATEGORY"]).agg(features_in_model=("FEATURE", "size"),
                                                      features_used=("MEAN_ABS_SHAP", lambda s: int((s > 0).sum())),
                                                      shap_share=("SHAP_SHARE", "sum"),
                                                      in_top_n=("SHAP_RANK", lambda s: int((s <= TOP_N).sum())))
               .reset_index().sort_values(["group", "shap_share"], ascending=[True, False]))
display(sub_summary)

fig, axes = plt.subplots(1, len(GROUPS), figsize=(16, 9))
for ax, g in zip(np.atleast_1d(axes), GROUPS):
    top = REPORT[g].head(25).iloc[::-1]
    ax.barh(top.FEATURE, top.MEAN_ABS_SHAP, color=np.where(top.SHAP_DIRECTION_RHO >= 0, "tab:blue", "tab:red"))
    ax.set_title(f"{CATEGORY}: {g} (blue = higher value, more NBRx)")
    ax.set_xlabel("weighted mean |SHAP| (log scale of NBRx)")
    ax.tick_params(axis="y", labelsize=7)
plt.tight_layout()
plt.show()

In [0]:
for g, st in RESULTS.items():
    sel = st[st.SELECTED]
    assert len(st) == len(FEATURES) and st.FEATURE.is_unique
    assert int(funnel[funnel.group == g].removed.sum()) + len(sel) == len(st), "stage counts do not add up"
    assert not sel.CONSTANT.any() and (sel.TOP_VALUE_SHARE < NEAR_CONSTANT_SHARE).all() and (sel.FILL_SHARE >= MIN_FILL_SHARE).all()
    assert KEPT_MAX_COPY_RHO[g] < NEAR_COPY_RHO, f"{g}: two kept features are near-copies"
    assert not (st.FLAG_5_NO_SIGNAL & st.FLAG_6_UNSTABLE).any(), "a feature has both flags"
    assert not (~st.SELECTED & (st.FLAG_5_NO_SIGNAL | st.FLAG_6_UNSTABLE)).any(), "a removed feature is flagged"
    assert set(MODELS[g]["features"]) == set(sel.FEATURE), "model features differ from the clean-up list"
    assert MODELS[g]["booster"].num_trees() == N_TREES, "model stopped early"
    assert SHAP_MAX_ERR[g] < 1e-3, f"{g}: SHAP values do not add up to the prediction"
    assert REPORT[g].FEATURE.is_unique and len(REPORT[g]) == len(sel)
assert len(disc) == len(FEATURES) and set(disc.FEATURE) == set(FEATURES)
assert group_summary.rows.sum() == len(keys) == X.shape[0]
print("All checks passed:", {g: f"{len(REPORT[g]):,} features in model, max kept copy rho {KEPT_MAX_COPY_RHO[g]:.3f}, "
                              f"SHAP error {SHAP_MAX_ERR[g]:.1e}" for g in GROUPS})

In [0]:
os.makedirs(OUT_DIR, exist_ok=True)
disc.to_csv(OUT_DIR + "mean_median_writers_vs_non_writers.csv", index=False)
funnel.to_csv(OUT_DIR + "cleanup_funnel.csv", index=False)
group_summary.to_csv(OUT_DIR + "group_summary.csv", index=False)
fit_summary.to_csv(OUT_DIR + "model_fit_in_sample.csv", index=False)
coverage.to_csv(OUT_DIR + "shap_coverage.csv", index=False)
sub_summary.to_csv(OUT_DIR + "shap_by_sub_category.csv", index=False)
dictionary.reset_index().to_csv(OUT_DIR + "feature_dictionary.csv", index=False)
for g in GROUPS:
    RESULTS[g].sort_values(["SELECTED", "SIGNAL_SCORE"], ascending=False).to_csv(f"{OUT_DIR}cleanup_stats_{g}.csv", index=False)
    REPORT[g].to_csv(f"{OUT_DIR}shap_ranking_{g}.csv", index=False)
    with open(f"{OUT_DIR}model_{g}.txt", "w") as f:   # model_to_string: Volumes do not support the append that save_model uses
        f.write(MODELS[g]["booster"].model_to_string())
    pd.Series(MODELS[g]["features"], name="FEATURE").to_csv(f"{OUT_DIR}model_{g}_feature_order.csv", index=False)   # f0, f1, ... in the model file
RUN_MINUTES = (time.time() - T_START) / 60
with open(OUT_DIR + "run_info.json", "w") as f:
    json.dump({"category": CATEGORY, "label": CATEGORY_LABEL, "run_name": RUN_NAME, "sample_list": SAMPLE_LIST, "cohorts": COHORT_LIST,
               "test_hcps": TEST_HCPS, "max_features": MAX_FEATURES, "rows": int(len(keys)), "features": len(FEATURES),
               "params": PARAMS, "n_trees": N_TREES, "noise_bars": THRESHOLDS, "run_minutes": RUN_MINUTES,
               "driver_cores": os.cpu_count(), "tables": [t["path"] for t in SOURCE_TABLES]}, f, indent=2, default=str)
print(f"Saved to {OUT_DIR} ({RUN_MINUTES:.1f} min in total)")
print(sorted(os.listdir(OUT_DIR)))